# Export Trained Model to ONNX
Run this AFTER training is complete.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

class DWSep(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class SE(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        m = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, m), nn.ReLU(True),
            nn.Linear(m, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

class MB(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        m = ic * 2
        self.ex = nn.Sequential(
            nn.Conv2d(ic, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.dw = DWSep(m, oc, st)
        self.se = SE(oc)
        self.res = (st == 1 and ic == oc)
    def forward(self, x):
        o = self.se(self.dw(self.ex(x)))
        return o + x if self.res else o

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True))
        self.blk = nn.Sequential(
            MB(64, 64), MB(64, 128, 2), MB(128, 128),
            MB(128, 256, 2), MB(256, 256), MB(256, 256),
            MB(256, 512, 2), MB(512, 512), MB(512, 512))
        self.fin = nn.Sequential(
            nn.Conv2d(512, 512, 3, groups=512, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, ed, 1, bias=False), nn.BatchNorm2d(ed))
    def forward(self, x):
        x = self.fin(self.blk(self.stem(x)))
        return F.normalize(x.view(x.size(0), -1), p=2, dim=1)

model = MobileFaceNet(128)
model.load_state_dict(torch.load('rec_best.pt', map_location='cpu'))
model.eval()
print('Model loaded from rec_best.pt')

!pip install -q onnxscript
torch.onnx.export(
    model, torch.randn(1, 3, 112, 112),
    'face_recognition.onnx',
    input_names=['input'],
    output_names=['output'],
    opset_version=13,
    dynamo=False)
sz = os.path.getsize('face_recognition.onnx') / 1e6
print('Exported: face_recognition.onnx (' + str(round(sz, 1)) + 'MB)')
print('Download and place in F:/PROJECTS/NetraEdge/models/')
